# AI Arena — Student Starter Notebook (all four tracks)

Welcome to the **D4 Summer 2026 AI Arena**. This notebook walks you through every track end-to-end — pick the section for the track you're working on, run the cells, and you'll have a real submission on the leaderboard within a few minutes.

## Tracks

| # | Track | Metric | Section |
|---|---|---|---|
| 1 | Hallucination Hunter | macro-F1 | [§1](#track-1) |
| 2 | Prompt Golf | judge accuracy × token efficiency | [§2](#track-2) |
| 3 | RAG Treasure Hunt | weighted rubric (30/25/15/15/10/5) | [§3](#track-3) |
| 4 | Build Your Own AI Judge | linear-weighted Cohen's κ → [0, 1] | [§4](#track-4) |

**Daily limits across all tracks:** 5 submissions per `student_id` per UTC day, plus 50,000 judge tokens for tracks that call the LLM judge (Track 2 & 3). The leaderboard keeps your **best** score across all submissions, Kaggle-style.

Full scoring rules: [`docs/judge_rubrics.md`](judge_rubrics.md) · FAQ: [`docs/faq.md`](faq.md)

## 0. Setup (run this first — every track uses it)

In [ ]:
%pip install -q httpx

import httpx, json, datetime, uuid

# --- Configure these ---
ARENA_URL  = 'https://d4-arena-api-77646749251.us-central1.run.app'  # production URL
STUDENT_ID = 'asukul'                                                # your Canvas/NetID
MODEL_USED = 'manual'                                                # 'claude-haiku-4-5', 'gemini-flash', 'manual', etc.

# Print the four /competitions/{track_id} URLs you'll be looking at while working.
for tid in ('hallucination_hunter', 'prompt_golf', 'rag_treasure_hunt', 'meta_judge'):
    print(f'  {ARENA_URL}/competitions/{tid}')

In [ ]:
# Common helper used by every track — submit, parse the response, then
# print where you landed on that track's leaderboard.

def submit_and_show(envelope: dict) -> None:
    track_id = envelope['track_id']
    with httpx.Client(timeout=30.0) as client:
        r = client.post(f'{ARENA_URL}/submit', json=envelope)

    if r.status_code == 200:
        body = r.json()
        print(f"Submitted! submissions_today = {body['submissions_today']}/{body['daily_limit']}")
    elif r.status_code == 422:
        print('Validation failed:')
        for p in r.json().get('problems', []):
            print(f"  - {p.get('field', '?')}: {p.get('problem', r.text)}")
        return
    elif r.status_code == 429:
        d = r.json()['detail']
        print(f"Daily limit reached ({d['current']}/{d['limit']}). Resets {d['reset_at']}.")
        return
    else:
        print(f'Unexpected status {r.status_code}: {r.text}')
        return

    # Best-effort: scoring is async, but the in-process queue scores synchronously
    # so the row is usually there by the time we read.
    leaderboard = httpx.get(f'{ARENA_URL}/leaderboard/{track_id}').json()
    print(f'\nTop 5 on {track_id}:')
    for rank, entry in enumerate(leaderboard['entries'][:5], start=1):
        marker = ' <-- you' if entry['student_id'] == STUDENT_ID else ''
        print(f"  {rank:>2}. {entry['student_id']:<20} {entry['final_score']:.4f}{marker}")


def fresh_envelope(track_id: str, *, prompt_version: str = 'v1', strategy: str = '') -> dict:
    return {
        'submission_id': f'sub_{STUDENT_ID}_{uuid.uuid4().hex[:8]}',
        'student_id':    STUDENT_ID,
        'track_id':      track_id,
        'submission_timestamp': datetime.datetime.now(datetime.UTC).isoformat(),
        'model_used':    MODEL_USED,
        'prompt_version': prompt_version,
        'self_reported_strategy': strategy,
        'track_payload': {},  # filled in per track below
    }

<a id="track-1"></a>
## §1 — Hallucination Hunter

Given a claim, predict whether it is `supported`, `refuted`, or `not_enough_info`.

**Scoring:** macro-F1 across the three labels (each weighted equally). Predicting one class for everything will not score well.

**No judge tokens** — this track is pure classification, so it costs nothing to iterate on.

In [ ]:
# Sample claims — replace with the real test set on launch day.
claims = [
    {'claim_id': 'C001', 'text': 'Iowa State University was founded in 1858.'},
    {'claim_id': 'C002', 'text': 'The Cy-Hawk trophy is awarded to the loser of the ISU vs Iowa football game.'},
    {'claim_id': 'C003', 'text': 'The Memorial Union has more square feet than the Parks Library.'},
    {'claim_id': 'C004', 'text': 'Iowa State established the first U.S. graduate program in artificial intelligence.'},
    {'claim_id': 'C005', 'text': 'The ISU mascot Cy was introduced in 1954.'},
]

def predict_label(claim_text: str) -> str:
    """Return one of 'supported', 'refuted', 'not_enough_info'.

    Replace this baseline with your own logic — keyword rules, an LLM call,
    a retrieval step, anything. The harness only cares about the label string.
    """
    return 'not_enough_info'  # baseline: predict the safe class for everything

In [ ]:
envelope = fresh_envelope(
    'hallucination_hunter',
    strategy='Baseline: always predict not_enough_info.',
)
envelope['track_payload'] = {
    'predictions': [
        {'claim_id': c['claim_id'], 'label': predict_label(c['text'])}
        for c in claims
    ],
}

submit_and_show(envelope)

<a id="track-2"></a>
## §2 — Prompt Golf

Find the shortest prompt that still answers correctly. You submit your `prompt_template` plus the (input, output) samples it produced when run against the gold inputs.

**Scoring:** `mean(judge_score) × min(1, baseline_tokens / total_tokens)`. Token efficiency saturates at 1.0 — accuracy is the ceiling, length is the tiebreaker.

**Judge tokens used:** ~one judge call per sample, against the per-day budget.

Tip: run your prompt against every gold input *before* submitting. The harness will reject submissions where the `samples` array doesn't cover all gold inputs.

In [ ]:
# Track 2's gold inputs are simple short-answer questions. The /samples
# endpoint always carries the canonical example — pull it so you can mutate
# from a known-good starting point.
canonical = httpx.get(f'{ARENA_URL}/samples/prompt_golf').json()
gold_inputs = [s['input'] for s in canonical['track_payload']['samples']]
print('Gold inputs you must cover:')
for q in gold_inputs:
    print(f'  - {q}')

In [ ]:
# --- Your prompt template + a function that runs it. ---
# The harness scores `mean(correctness) * min(1, baseline / total_tokens)`,
# so a shorter prompt that still scores 0.9 beats a long one at 0.9.

PROMPT_TEMPLATE = 'Answer concisely:'

def run_my_prompt(input_text: str) -> str:
    """Run PROMPT_TEMPLATE + input_text through your model and return the answer.

    Replace this with a real model call (Claude, Gemini, OpenAI, local model).
    For testing without an API key, hand-craft answers below.
    """
    # Hand-crafted baseline so the notebook runs end-to-end without an API key.
    answers = {
        'What is 2+2?':                '4',
        'Capital of France?':          'Paris',
        "Who wrote 'Hamlet'?":         'William Shakespeare',
    }
    return answers.get(input_text, '(unknown)')

samples = [
    {'input': q, 'output': run_my_prompt(q)} for q in gold_inputs
]
for s in samples:
    print(f"  {s['input']!r:35} → {s['output']!r}")

In [ ]:
envelope = fresh_envelope(
    'prompt_golf',
    strategy='Terse system prompt; covers the three gold questions.',
)
envelope['track_payload'] = {
    'prompt_template': PROMPT_TEMPLATE,
    'samples': samples,
}

submit_and_show(envelope)

<a id="track-3"></a>
## §3 — RAG Treasure Hunt

Build a retrieval-augmented generation pipeline over the public ISU course catalog corpus. For each question, retrieve relevant chunks, ground your answer in them, and cite the chunks you used.

**Scoring (weighted rubric):**

| Dimension | Weight | What it measures |
|---|---|---|
| Correctness | 30% | LLM-judged answer quality vs. gold |
| Faithfulness | 25% | LLM-judged: is the answer grounded in your cited chunks? |
| Retrieval | 15% | Recall of gold-relevant chunks in your `retrieved_contexts` |
| Citations | 15% | F1 of your `citations` vs. gold-citation set |
| Cost | 10% | Lower `estimated_cost_usd` than baseline → 1.0 |
| Safety | 5% | LLM-judged: no harmful / out-of-scope content |

Public corpus: [`corpora/isu_course_catalog/index.json`](https://github.com/asukul/AI-arena/blob/main/corpora/isu_course_catalog/index.json) · public questions: [`questions.json`](https://github.com/asukul/AI-arena/blob/main/corpora/isu_course_catalog/questions.json).

In [ ]:
# Pull the canonical sample as a starting point — it's the shape of a perfect
# submission. Read the chunks, then write your own answers + citations.
canonical = httpx.get(f'{ARENA_URL}/samples/rag_treasure_hunt').json()
for ans in canonical['track_payload']['answers'][:2]:
    print(f"Q {ans['question_id']}:")
    print(f"  answer: {ans['answer'][:90]}…")
    print(f"  cited: {ans['citations']}, retrieved: {ans['retrieved_contexts']}")
    print(f"  cost: ${ans['estimated_cost_usd']:.4f}, latency: {ans['latency_seconds']}s")

In [ ]:
# --- Build your retrieval + generation pipeline here. ---
# A reasonable baseline:
#   1. Embed every chunk in index.json once (or use BM25 — both work).
#   2. For each question, retrieve top-k chunks.
#   3. Pass them + the question to an LLM with a strict 'cite chunk_id N' rule.
#   4. Parse the cited chunk_ids out of the LLM output → `citations`.
#   5. Track every chunk_id you considered → `retrieved_contexts` (recall is graded too!).
#
# This cell uses the canonical sample as-is so the notebook runs end-to-end.
# Replace with your own pipeline.

answers = canonical['track_payload']['answers']
print(f'Built {len(answers)} answers; {sum(len(a["citations"]) for a in answers)} citations total.')

In [ ]:
envelope = fresh_envelope(
    'rag_treasure_hunt',
    strategy='Hybrid BM25 + dense retrieval, top-3 chunks per question.',
)
envelope['track_payload'] = {'answers': answers}

submit_and_show(envelope)

<a id="track-4"></a>
## §4 — Build Your Own AI Judge

The capstone. You write a small evaluator function plus the rubric text it implements, and grade a calibration set of 10 items on a 1–5 ordinal scale. We score how well your grades agree with the instructor's gold ratings using **linear-weighted Cohen's κ**.

**Scoring:** `final_score = (κ + 1) / 2` — perfect agreement = 1.0, chance = 0.5, reverse agreement = 0.

**Linear weighting** means an off-by-one rating is much better than off-by-three — your function doesn't have to be perfect, just *consistently directional*.

**No judge tokens** — your evaluator is the judge. The platform doesn't execute your code; the source is grading-only material.

In [ ]:
# --- Your rubric + evaluator. ---
# The rubric is shown to a human grader; the evaluator is the function that
# produces your grades. Both are part of the submission.

RUBRIC_TEXT = '''
Score each response on a 1-5 ordinal scale:
  5 = answers correctly and cites all relevant sources.
  4 = answers correctly with minor omissions.
  3 = partially correct or missing a citation.
  2 = mostly wrong but on-topic.
  1 = irrelevant, empty, or harmful.
'''

EVALUATOR_SOURCE = '''
def grade(item: dict) -> int:
    """Return a 1-5 ordinal rating."""
    text = item["response"].lower()
    if not text.strip():
        return 1
    keyword_hits = sum(1 for k in item["keywords"] if k.lower() in text)
    return max(1, min(5, keyword_hits))
'''

# Mirror EVALUATOR_SOURCE so we can run it locally to produce self_grades.
exec(EVALUATOR_SOURCE)

In [ ]:
# In production you'd grade the calibration set the instructor releases.
# For the canonical run, post the same self_grades the platform's gold expects.
canonical = httpx.get(f'{ARENA_URL}/samples/meta_judge').json()
self_grades = canonical['track_payload']['self_grades']
for sg in self_grades[:3]:
    print(sg)
print(f'… {len(self_grades)} grades total.')

In [ ]:
envelope = fresh_envelope(
    'meta_judge',
    strategy='Keyword-presence rubric, integer 1-5 ratings.',
)
envelope['track_payload'] = {
    'evaluator_source': EVALUATOR_SOURCE,
    'rubric_text':      RUBRIC_TEXT,
    'self_grades':      self_grades,
}

submit_and_show(envelope)

## Tips for every track

- **Use `/samples/{track_id}` as scaffolding.** The canonical sample for every track scores 1.0 against the in-tree gold — start there, mutate, learn the scoring rules quickly.
- **Don't burn submissions on debugging.** You only get 5/day. Validate your envelope locally first (matching `submission_id`/`student_id`, every gold ID covered, no extra fields) before posting.
- **Watch your judge-token budget on Tracks 2 & 3.** 50,000 tokens/day. The dashboard prints `tokens_used_today` in the response when you're over half.
- **Best score wins.** A bad submission doesn't push your good one off the leaderboard — submit experimental approaches without fear.
- **Read your per-dimension breakdown.** `judge_metadata.dimensions` shows where you're weak (e.g. high correctness, low faithfulness on Track 3) so you know what to fix.
- **`model_used` and `self_reported_strategy` matter.** They aren't graded, but the cohort retrospectives at the end of the bootcamp will lean heavily on them — be honest and specific.

Good luck.